In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("../data/processed/master_mile12_data.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.head()

In [ ]:
target_cols = [
    'tomato_price_ngn', 'onion_price_ngn', 
    'petrol_price_ngn', 'diesel_price_ngn', 
    'north_rainfall_mm', 'usd_ngn_rate'
]

# Generate descriptive statistics
stats = df[target_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
stats.rename(columns={'50%': 'median'}, inplace=True)
display(stats)

# Check correlations
corr_matrix = df[target_cols].corr()
print(corr_matrix)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['date'], df['tomato_price_ngn'], label='Tomato Price (NGN)', color='tomato')
plt.plot(df['date'], df['onion_price_ngn'], label='Onion Price (NGN)', color='purple')
plt.plot(df['date'], df['diesel_price_ngn'], label='Diesel Price (NGN)', color='gray', linestyle='--')
plt.plot(df['date'], df['usd_ngn_rate'] * 1000, label='USD/NGN Rate (Scaled)', color='blue', linestyle=':')

# Vertical marker for June 2023 shock (subsidy removal & FX float)
shock_date = pd.to_datetime('2023-06-01')
plt.axvline(x=shock_date, color='red', linestyle='-', label='Macro Shock (June 2023)')

plt.title('Tomato & Onion Prices vs. Macroeconomic Shocks (2020–2025)')
plt.xlabel('Date')
plt.ylabel('Value / Price (NGN)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Pre vs Post shock average comparison
pre_shock = df[df['date'] < shock_date][target_cols].mean()
post_shock = df[df['date'] >= shock_date][target_cols].mean()
shock_comparison = pd.DataFrame({'Pre-Shock Avg': pre_shock, 'Post-Shock Avg': post_shock})
display(shock_comparison)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x=df['date'].dt.month, y=df['tomato_price_ngn'], ax=axes[0], palette='Reds')
axes[0].set_title('Tomato Price Seasonality')
axes[0].set_xlabel('Month')

sns.boxplot(x=df['date'].dt.month, y=df['onion_price_ngn'], ax=axes[1], palette='Purples')
axes[1].set_title('Onion Price Seasonality')
axes[1].set_xlabel('Month')

sns.boxplot(x=df['date'].dt.month, y=df['north_rainfall_mm'], ax=axes[2], palette='Blues')
axes[2].set_title('Northern Rainfall Seasonality')
axes[2].set_xlabel('Month')

plt.tight_layout()
plt.show()

In [ ]:
# Create lag copies for heatmap analysis
df_corr = df[target_cols].copy()
df_corr['tomato_lag1'] = df['tomato_price_ngn'].shift(1)
df_corr['tomato_lag2'] = df['tomato_price_ngn'].shift(2)
df_corr['onion_lag1'] = df['onion_price_ngn'].shift(1)
df_corr['onion_lag2'] = df['onion_price_ngn'].shift(2)

plt.figure(figsize=(10, 8))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Same-Month & Lagged Correlation Heatmap')
plt.show()